# GRASP on MUCH — end to end

Training-free, claim-level, attention-based hallucination detection.

**Runtime → Change runtime type → T4 GPU** before running anything.

1. mount Drive, point at the project
2. set up MUCH (installs deps, clones, patches for Python 3.11)
3. HuggingFace login (Llama and Gemma are gated)
4. download MUCH + the precomputed baseline signals
5. **checkpoint** — reproduce CCP's published number. If this fails, stop.
6. extract attention features (the only GPU step)
7. score, sweep, ablate, compare

Step 5 is a genuine go/no-go. Do not skip it.

## 1 · Project location

In [ ]:
!nvidia-smi -L

import os, sys, json, time
from pathlib import Path
ROOT = Path('/content'); os.chdir(ROOT)

from google.colab import drive; drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ai_project')   # upload ai_project here
assert PROJECT.exists(), f'ai_project not found at {PROJECT}'
sys.path.insert(0, str(PROJECT/'src'))
print('project :', PROJECT)
print('python  :', sys.version.split()[0])

## 2 · Set up MUCH

This installs the dependencies, clones the benchmark and patches it.

Two things it fixes that will otherwise waste your afternoon:

- `src/__init__.py` imports **every** submodule, so importing anything drags in
  the annotation and generation stacks. The full import-time dependency set is
  installed, not just the obvious ones. (`python-decouple` is the first one that
  bites.)
- MUCH targets **Python 3.12**. Two f-strings in `annotation/human_evaluator.py`
  put a backslash inside the expression part, which is legal only on 3.12+. On a
  3.11 runtime the whole package dies with a SyntaxError in a file you never
  meant to touch. The patch hoists those expressions out; it is a no-op on 3.12
  and safe to re-run.

Restart the runtime afterwards if pip replaced anything already imported.

In [ ]:
!python {PROJECT}/scripts/01_setup_much.py --root /content

In [ ]:
# Llama and Gemma are gated. Accept the licences on their model pages first,
# then paste a read token.
from huggingface_hub import login
login()

## 3 · Download MUCH and the precomputed baselines

`import-signals` (not `import-baselines`) is what pulls the baseline signals, so
CCP does not have to be recomputed on an A100 you do not have.

In [ ]:
MUCH = ROOT/'much'
os.chdir(MUCH)
!python -m src import-much
!python -m src import-signals
!ls -la output && ls -la output/signals 2>/dev/null | head -20

In [ ]:
sys.path.insert(0, str(MUCH))
from src.annotation import get_all_annotations
from src.generation import get_all_generations

anns = get_all_annotations(); gens = get_all_generations()
print(f'{len(anns)} annotations, {len(gens)} generations')

from collections import Counter
print('by lang  :', Counter(g.lang for g in gens.values()))
print('by model :', Counter(g.generation_cfg.model_name for g in gens.values()))

# NOTE: never print an Annotation directly. Its __repr__ decodes claim text,
# which loads the tokenizer, which hits HuggingFace -- and dies with a 403 if
# your account is not yet authorised for that model. Inspect it tokenizer-free.
a = list(anns.values())[0]; g = a.generation
print(f'\\nsample {g.id}  |  {g.generation_cfg.model_name}  |  lang={g.lang}')
print('prompt :', g.prompt)
print('output :', g.output[:300].replace(chr(10), ' '), '...')
print(f'claims : {len(a.token_chunks)}')
for i, c in enumerate(a.token_chunks):
    labs = {k: v[i] for k, v in a.labels.items() if i < len(v)}
    print(f'  #{i:2}  tokens[{c[0]}:{c[-1]+1}]  labels={labs}')

### Which models can you actually load?

Llama and Gemma are **gated**: a token is not enough, your account has to be
approved on each model's page. Check now rather than finding out mid-extraction.

You only need the models you actually extract from. Start with
**Llama-3.2-3B-Instruct**; request the rest later. The CCP checkpoint below needs
no model at all, so a BLOCKED line here does not stop you proceeding.

In [ ]:
from huggingface_hub import model_info

for m in sorted({g.generation_cfg.model_name for g in gens.values()}):
    try:
        model_info(m)
        print(f'  OK      {m}')
    except Exception as exc:
        print(f'  BLOCKED {m}  ({type(exc).__name__})')
        print(f'          request access at https://huggingface.co/{m}')

## 4 · Checkpoint — reproduce the CCP baseline

This runs MUCH's **exact** evaluation protocol. Four rules make or break it, and
all four are easy to get wrong:

1. Labels are `{-1, +1}` and **`-1` is the hallucination class**. Reading them as
   0/1 inverts the positive class and hands you an AUROC below 0.5.
2. The **trailing EOS chunk is dropped** — MUCH iterates `token_chunks[:-1]`.
3. **Stopwords and punctuation are zeroed** before aggregating. This needs each
   generating model's tokenizer, so gated-model access *is* required here.
4. Only annotations where gpt-4o and gpt-4.1 **agree** are scored.

These live in `grasp/much_eval.py`, which is cross-checked against MUCH's own
code by `src/tests/test_much_eval.py` — the aggregators match to 1e-12.

Published reference: **ROC-AUC 0.772 / PR-AUC 0.639**.

**Paste this cell's output back into the chat.**

In [ ]:
# Sanity-check our transcription of MUCH's evaluator before trusting any number.
!python {PROJECT}/src/tests/test_much_eval.py --much-root {MUCH}

In [ ]:
# Checkpoint. Defaults to ccp_24_8 -- MUCH's own docstring specifies
# delta=24, sigma=8, so that is the variant behind the published 0.772.
!python {PROJECT}/scripts/04_reproduce_ccp.py --much-root {MUCH} \
    --skip-unavailable --out {PROJECT}/results/ccp_reproduction.json

In [ ]:
# The full baseline table -- all 13 shipped signals, every language, every model.
# A few minutes, CPU only. This IS Table 1 of the paper, so keep the JSON.
!python {PROJECT}/scripts/04_reproduce_ccp.py --much-root {MUCH} \
    --all-signals --skip-unavailable \
    --out {PROJECT}/results/all_baselines.json

In [ ]:
# If a tokenizer is gated and you cannot wait, this skips rule 3.
# It is a PLUMBING CHECK ONLY — the numbers will not match the published table.
# !python {PROJECT}/scripts/04_reproduce_ccp.py --much-root {MUCH} \
#     --no-stopword-mask --out {PROJECT}/results/ccp_nomask.json

## 5 · Extract attention features (GPU)

One generation config at a time. Start with **Llama-3.2-3B** — smallest model,
standard causal attention, fastest turnaround. Leave Gemma-3 for last: its
interleaved sliding-window attention complicates the prompt/generation split.

In [ ]:
cfgs = sorted({g.generation_config for g in gens.values()})
for c in cfgs:
    ex = next(g for g in gens.values() if g.generation_config == c)
    n = sum(1 for g in gens.values() if g.generation_config == c)
    print(f'{c}  |  {ex.generation_cfg.model_name}  |  T={ex.generation_cfg.temperature}  |  {n} gens')

In [ ]:
# Pick by MODEL NAME, not by list position -- cfgs[0] happens to be
# Ministral-8B, which is bigger and slower than what we want first.
TARGET_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
TARGET_TEMP  = 0.7

def cfg_for(model, temp):
    for c in cfgs:
        ex = next(g for g in gens.values() if g.generation_config == c)
        if ex.generation_cfg.model_name == model and \
           abs(ex.generation_cfg.temperature - temp) < 1e-6:
            return c
    raise SystemExit(f'no config for {model} at T={temp}')

CONFIG = cfg_for(TARGET_MODEL, TARGET_TEMP)
CACHE  = PROJECT/'results'/f'cache_{CONFIG}'
print('config:', CONFIG, '->', TARGET_MODEL, 'T=', TARGET_TEMP)

# Smoke test on 20 generations. If selected_heads comes back empty,
# eager attention is not on and nothing was captured.
!python {PROJECT}/scripts/02_extract_attention.py \
    --much-root {MUCH} --config {CONFIG} --out {CACHE} --calib 10 --limit 20

In [ ]:
# Full pass. Resumable — cached generations are skipped, so if Colab drops just
# re-run this cell.
!python {PROJECT}/scripts/02_extract_attention.py \
    --much-root {MUCH} --config {CONFIG} --out {CACHE} --calib 200

print(json.loads((CACHE/'timing.json').read_text()))
print(json.loads((CACHE/'selected_heads.json').read_text()))

## 6 · Score, sweep, ablate

CPU only and fast, so iterate freely. Sweep on the training split, then freeze
the configuration before touching the human-annotated test split.

In [ ]:
!python {PROJECT}/scripts/03_score_and_eval.py \
    --much-root {MUCH} --cache {CACHE} --lang en --sweep \
    --out {PROJECT}/results/grasp_en_sweep.json

In [ ]:
TAU, ALPHA = 8.0, 0.6      # <- paste the winning values from the sweep

!python {PROJECT}/scripts/03_score_and_eval.py \
    --much-root {MUCH} --cache {CACHE} --lang en \
    --tau {TAU} --alpha {ALPHA} --ablate --fuse-with {SIG} \
    --out {PROJECT}/results/grasp_en_final.json

print(json.dumps(json.loads((PROJECT/'results'/'grasp_en_final.json').read_text()), indent=2))

In [ ]:
# All four languages with the frozen configuration.
for lang in ('en','fr','es','de'):
    !python {PROJECT}/scripts/03_score_and_eval.py \
        --much-root {MUCH} --cache {CACHE} --lang {lang} \
        --tau {TAU} --alpha {ALPHA} \
        --out {PROJECT}/results/grasp_{lang}.json

## 7 · The runtime column

Half the contribution. MUCH's own accounting: generating the 4,873 samples took
**2,758 s** on an A100, and CCP cost **3,230–5,429 s** on the same hardware —
roughly **124% overhead**.

Report the *ratio* to generation time measured on **your** GPU, not raw seconds.
That is what makes the comparison fair, and the benchmark asks for it explicitly.

In [ ]:
print(json.loads((CACHE/'timing.json').read_text()))
print('\nDenominator — run MUCH\'s own generation timer on this same GPU:')
!cd {MUCH} && python -m src evaluate-generation-time --approx-factor 0.05